# Reentrenamiento del detector de placas con YOLOv8 en Colab

Este cuaderno está preparado para entrenar un modelo YOLOv8 usando datos en formato YOLO (imágenes + etiquetas + `data.yaml`) desde Google Drive.

### Requisitos
- Tener la carpeta del dataset en Drive
- Tener una estructura tipo: `train/` y `val/` con imágenes y etiquetas
- Ajustar las rutas según la ubicación real de tu proyecto

> Este notebook puede usarse como reemplazo del entrenamiento local para volver a entrenar el detector de placas.

In [ ]:
# 1) Montar Google Drive
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# Rutas probables según la estructura de tu Drive
possible_roots = [
    Path('/content/drive/MyDrive/Deployment'),
    Path('/content/drive/MyDrive/Colab Notebooks/Deployment'),
    Path('/content/drive/MyDrive/Colab Notebooks') / 'Deployment'
]

PROJECT_ROOT = next((p for p in possible_roots if p.exists()), None)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'No encontré la carpeta del proyecto. Revisa si está en MyDrive/Deployment o en MyDrive/Colab Notebooks/Deployment.'
    )

print('Proyecto:', PROJECT_ROOT)
print('Contenido de MyDrive:')
!ls -la '/content/drive/MyDrive' | head -n 20

# Verifica la carpeta exacta encontrada
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'No existe la carpeta del proyecto en {PROJECT_ROOT}.')


In [ ]:
# 2) Instalar dependencias compatibles con Colab y reiniciar el kernel
!pip install -q "numpy<2.0.0" ultralytics pyyaml

import os
os.kill(os.getpid(), 9)

In [ ]:
# 2b) Verificar dependencias instaladas
import ultralytics, torch, cv2, numpy as np
print('✅ Dependencias instaladas y verificadas:')
print('  ultralytics:', ultralytics.__version__)
print('  numpy:', np.__version__)
print('  torch:', torch.__version__)

In [5]:
# 3) Preparar rutas del proyecto y del dataset
from pathlib import Path
import os

# Detección automática de la ruta real del dataset
candidates = [
    PROJECT_ROOT,
    PROJECT_ROOT / 'data',
    PROJECT_ROOT / 'dataset',
    PROJECT_ROOT / 'Dataset',
]

DATASET_DIR = next((p for p in candidates if p and p.exists() and (p / 'train').exists() and (p / 'val').exists()), None)

# Búsqueda alternativa si no se encuentra en las rutas candidatas directas
if not DATASET_DIR and PROJECT_ROOT and PROJECT_ROOT.exists():
    for p in PROJECT_ROOT.rglob('*'):
        if p.is_dir() and (p / 'train').exists() and (p / 'val').exists():
            DATASET_DIR = p
            break

if not DATASET_DIR or not DATASET_DIR.exists():
    DATASET_DIR = PROJECT_ROOT
    print(f'⚠️ Advertencia: No se detectó automáticamente una carpeta con train y val. Usando {DATASET_DIR}')
else:
    print(f'✅ Dataset detectado en: {DATASET_DIR}')

MODELS_DIR = PROJECT_ROOT / 'backend' / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('📁 Ruta de Dataset:', DATASET_DIR)
print('📁 Ruta de Modelos:', MODELS_DIR)
print('-' * 50)

img_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

for folder in ['train', 'val', 'test']:
    p = DATASET_DIR / folder
    if not p.exists():
        if folder != 'test':
            print(f'❌ No existe {folder}, revisa la estructura del dataset.')
        continue
    
    print(f'✅ {folder} existe')
    for sub in ['images', 'labels']:
        subdir = p / sub
        if subdir.exists():
            if sub == 'images':
                files = [f for f in subdir.glob('*') if f.suffix.lower() in img_exts]
            else:
                files = list(subdir.glob('*.txt'))
            print(f'   └── 📁 {sub}: {len(files)} archivos encontrados')
        else:
            files = [f for f in p.glob('*') if f.suffix.lower() in img_exts] if sub == 'images' else list(p.glob('*.txt'))
            if files:
                print(f'   └── ⚠️ Subcarpeta {sub} no existe, pero hay {len(files)} archivos directamente en {folder}/')
            else:
                print(f'   └── ❌ No existe {sub} dentro de {folder}')


✅ Dataset detectado en: /content/drive/MyDrive/Deployment
📁 Ruta de Dataset: /content/drive/MyDrive/Deployment
📁 Ruta de Modelos: /content/drive/MyDrive/Deployment/backend/models
--------------------------------------------------
✅ train existe
   └── 📁 images: 4554 archivos encontrados
   └── 📁 labels: 4554 archivos encontrados
✅ val existe
   └── 📁 images: 803 archivos encontrados
   └── 📁 labels: 803 archivos encontrados


In [6]:
# 4) Crear archivo data.yaml
from pathlib import Path
import yaml

train_dir = DATASET_DIR / 'train'
val_dir = DATASET_DIR / 'val'

if not train_dir.exists() or not val_dir.exists():
    raise FileNotFoundError(f'No se encontró train/val dentro de {DATASET_DIR}. Revisa la ruta.')

# Estructura típica: train/images, train/labels, val/images, val/labels
train_images = train_dir / 'images'
val_images = val_dir / 'images'
train_labels = train_dir / 'labels'
val_labels = val_dir / 'labels'

if not train_images.exists() or not train_labels.exists():
    raise FileNotFoundError(f'Falta train/images o train/labels dentro de {train_dir}.')
if not val_images.exists() or not val_labels.exists():
    raise FileNotFoundError(f'Falta val/images o val/labels dentro de {val_dir}.')

# YOLO usa la ruta de las imágenes; las etiquetas deben estar en el sibling labels/.
dataset_yaml = {
    'train': str(train_images),
    'val': str(val_images),
    'nc': 1,
    'names': ['plate']
}

yaml_path = PROJECT_ROOT / 'data.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(dataset_yaml, f, sort_keys=False)

print('data.yaml generado en:', yaml_path)
print(yaml.safe_dump(dataset_yaml, sort_keys=False))

# Verificar que haya imágenes reales
for split_name, split_path in [('train', train_images), ('val', val_images)]:
    files = list(split_path.rglob('*'))
    image_files = [f for f in files if f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}]
    print(f'{split_name}: {len(image_files)} imágenes encontradas')
    if not image_files:
        print(f'Advertencia: no se encontraron imágenes en {split_path}')


data.yaml generado en: /content/drive/MyDrive/Deployment/data.yaml
train: /content/drive/MyDrive/Deployment/train/images
val: /content/drive/MyDrive/Deployment/val/images
nc: 1
names:
- plate

train: 4554 imágenes encontradas
val: 803 imágenes encontradas


In [7]:
# 5) Entrenar el modelo YOLOv8
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    name='plate_detector_yolov8',
    project=str(PROJECT_ROOT / 'runs'),
    exist_ok=True,
    patience=15,
    optimizer='auto',
    seed=42,
    verbose=True
)

print('Entrenamiento finalizado')
print('Resultado:', results)


Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Deployment/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=plate_detector_yolov8, nbs=64, nms

In [8]:
# 6) Validar y exportar el modelo
from pathlib import Path

best_model = PROJECT_ROOT / 'runs' / 'detect' / 'plate_detector_yolov8' / 'weights' / 'best.pt'
print('Ruta del mejor modelo:', best_model)

# Validación del mejor modelo
if best_model.exists():
    model = YOLO(str(best_model))
    metrics = model.val(data=str(yaml_path), imgsz=640)
    print(metrics)
else:
    print('No se encontró el mejor modelo entrenado; revisa la carpeta runs/.')

# Exportar a ONNX para uso más ligero
try:
    exported_model = model.export(format='onnx', imgsz=640, half=False)
    print('Modelo exportado:', exported_model)
except Exception as e:
    print('No se pudo exportar a ONNX:', e)

# Copiar el mejor modelo al proyecto
final_model_path = MODELS_DIR / 'best.pt'
if best_model.exists():
    !cp "{best_model}" "{final_model_path}"
    print('Modelo copiado a:', final_model_path)
else:
    print('No se encontró el mejor modelo entrenado.')


Ruta del mejor modelo: /content/drive/MyDrive/Deployment/runs/detect/plate_detector_yolov8/weights/best.pt
No se encontró el mejor modelo entrenado; revisa la carpeta runs/.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/Deployment/runs/plate_detector_yolov8/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.9 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 12 packages in 381ms
Prepared 5 packages in 1.32s
Uninstal